# Smart Packaging and Remaining Shelf Life (RSL) Prediction of White Button Mushrooms
## Multimodal Deep Learning (Vision + Packaging Treatment Formulation)
**Research Project:** Master's Food Science & Computer Vision Research
**Objective:** Predict the continuous **Remaining Shelf Life (RSL in Days)** of packaged white button mushrooms.
Features:
- Multimodal deep learning architecture fusing visual cap features with active packaging formulations.
- Heavy data augmentation (Flips, Affine, ColorJitter, RandomErasing) tailored to prevent overfitting on small samples.
- Zero data leakage via strict **Sample-ID Partitioning**.
- Benchmark comparison: **Multimodal Swin Transformer** vs. **Multimodal ResNet50**.
- Regression Metrics: **MAE (days)**, **RMSE (days)**, and **R² Score**.


In [ ]:
import os
import sys
import time
import zipfile
import shutil
import re
import json
from pathlib import Path

import torch
import torch.nn as nn
from torchvision import transforms
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
import numpy as np
import cv2
import matplotlib.pyplot as plt
from PIL import Image

print('=' * 70)
print('ENVIRONMENT & HARDWARE ACCELERATION CHECK')
print('=' * 70)
print(f'PyTorch Version:    {torch.__version__}')
print(f'CUDA Available:     {torch.cuda.is_available()}')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Active Device:      {device}')
if device.type == 'cuda':
    print(f'GPU Device Name:    {torch.cuda.get_device_name(0)}')
    print(f'GPU Memory Total:   {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB')
print('=' * 70)


### 1. Dataset Extraction & Directory Resolution
Extracts the preprocessed dataset archive (`mushroom_preprocessed_224.zip`) into `/kaggle/working/dataset/`.


In [ ]:
DATASET_DIR = Path('/kaggle/working/dataset')
DATASET_DIR.mkdir(parents=True, exist_ok=True)

# Search for zip in Kaggle input or local directory
zip_candidates = list(Path('/kaggle/input').glob('**/*mushroom*.zip')) + list(Path('.').glob('**/*mushroom*.zip'))

if zip_candidates:
    chosen_zip = zip_candidates[0]
    print(f'Found dataset archive: {chosen_zip}')
    with zipfile.ZipFile(chosen_zip, 'r') as zf:
        zf.extractall(DATASET_DIR)
    print(f'Extracted dataset to: {DATASET_DIR}')
else:
    pre_candidates = list(Path('/kaggle/input').glob('**/Control'))
    if pre_candidates:
        DATASET_DIR = pre_candidates[0].parent
        print(f'Using pre-mounted dataset directory: {DATASET_DIR}')
    else:
        print('[Warning] No zip found in /kaggle/input. Using preprocessed_224 if present.')
        DATASET_DIR = Path('preprocessed_224').resolve()

treatments = [d.name for d in DATASET_DIR.iterdir() if d.is_dir() and not d.name.startswith('.')]
print(f'Discovered treatments: {treatments}')
image_count = len(list(DATASET_DIR.glob('*/*/*.*')))
print(f'Total preprocessed images available: {image_count} (Expected: 792)')


### 2. Remaining Shelf Life (RSL) Formulation & Multimodal Dataset
- **Commercial Usability Endpoints:**
  - Control (Uncoated): Spoilage onset is Day 7 $ightarrow$ Shelf life = 7 days.
  - Active Packaging (1SKEO.2CEO & 2SKEO.1CEO): Spoilage onset delayed to Day 20 $ightarrow$ Shelf life = 20 days.
- **Continuous Target:** $RSL = \max(0, 	ext{Threshold} - 	ext{Current Day})$.
- **Multimodal Encoding:** Treatment formulation is encoded as a One-Hot vector `[Control, 1SKEO.2CEO, 2SKEO.1CEO]`.
- **Advanced Augmentation:** Flips, Rotations (±25°), Affine transforms, ColorJitter, and RandomErasing.


In [ ]:
# RSL Thresholds based on biological spoilage onset (Class 2)
SPOILAGE_THRESHOLDS = {
    'Control': 7.0,
    '1SKEO.2CEO': 20.0,
    '2SKEO.1CEO': 20.0,
}

TREATMENT_TO_INDEX = {
    'Control': 0,
    '1SKEO.2CEO': 1,
    '2SKEO.1CEO': 2,
}

def calculate_rsl(treatment, current_day):
    thresh = SPOILAGE_THRESHOLDS.get(treatment, 7.0)
    return float(max(0.0, thresh - current_day))

def get_splits(data_dir):
    split_file = data_dir / 'splits.json'
    if split_file.exists():
        with open(split_file, 'r') as f:
            return json.load(f)
    return {
        'train': [2, 3, 5, 6, 7, 10, 11, 12, 13, 14, 15, 16, 17, 19, 20, 22],
        'val': [8, 9, 18],
        'test': [1, 4, 21]
    }

class MushroomRSLDataset(torch.utils.data.Dataset):
    def __init__(self, root_dir, split='train', transform=None):
        self.root_dir = Path(root_dir)
        self.split = split.lower()
        self.transform = transform
        self.splits = get_splits(self.root_dir)
        self.allowed_ids = set(self.splits[self.split]) if self.split != 'all' else set(range(1, 23))
        self.samples = self._load_samples()

    def _load_samples(self):
        samples = []
        pattern = re.compile(r'^(\d+)-([bBsS])\.')
        for img_path in sorted(self.root_dir.glob('*/*/*.*')):
            if img_path.suffix.lower() not in ('.jpg', '.jpeg', '.png'):
                continue
            t_name = img_path.parent.parent.name
            if t_name not in TREATMENT_TO_INDEX:
                continue
            day = int(img_path.parent.name) if img_path.parent.name.isdigit() else -1
            if day < 0:
                continue
            m = pattern.match(img_path.name)
            if not m:
                continue
            s_id = int(m.group(1))
            if s_id in self.allowed_ids:
                rsl = calculate_rsl(t_name, day)
                samples.append({
                    'path': str(img_path),
                    'treatment': t_name,
                    'treatment_idx': TREATMENT_TO_INDEX[t_name],
                    'day': day,
                    'sample_id': s_id,
                    'rsl': rsl
                })
        return samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        item = self.samples[idx]
        image = Image.open(item['path']).convert('RGB')
        if self.transform:
            image = self.transform(image)

        treatment_one_hot = torch.zeros(3, dtype=torch.float32)
        treatment_one_hot[item['treatment_idx']] = 1.0

        target_rsl = torch.tensor([item['rsl']], dtype=torch.float32)
        return image, treatment_one_hot, target_rsl

# Advanced Augmentation Pipeline
train_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=25),
    transforms.RandomAffine(degrees=0, translate=(0.06, 0.06), scale=(0.92, 1.08)),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.25, scale=(0.02, 0.2), value='random')
])

eval_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_ds = MushroomRSLDataset(DATASET_DIR, split='train', transform=train_tf)
val_ds = MushroomRSLDataset(DATASET_DIR, split='val', transform=eval_tf)
test_ds = MushroomRSLDataset(DATASET_DIR, split='test', transform=eval_tf)

train_loader = torch.utils.data.DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=2, pin_memory=True)
val_loader = torch.utils.data.DataLoader(val_ds, batch_size=16, shuffle=False, num_workers=2, pin_memory=True)
test_loader = torch.utils.data.DataLoader(test_ds, batch_size=16, shuffle=False, num_workers=2, pin_memory=True)

print(f'Train Samples: {len(train_ds)} (16 IDs) | Val Samples: {len(val_ds)} (3 IDs) | Test Samples: {len(test_ds)} (3 IDs)')


### 3. Multimodal Regression Architectures (Swin-T vs. ResNet50)
Fuses visual degradation representation with treatment packaging formulation to produce continuous RSL predictions.


In [ ]:
from torchvision.models import resnet50, ResNet50_Weights, swin_t, Swin_T_Weights

class MultimodalSwinRSL(nn.Module):
    def __init__(self, pretrained=True, treatment_dim=3, emb_dim=32, dropout=0.3):
        super().__init__()
        self.model_name = 'multimodal_swin_t'
        weights = Swin_T_Weights.IMAGENET1K_V1 if pretrained else None
        base_swin = swin_t(weights=weights)

        self.features = base_swin.features
        self.norm = base_swin.norm
        self.permute = base_swin.permute
        self.avgpool = base_swin.avgpool
        vision_dim = 768

        self.treatment_net = nn.Sequential(
            nn.Linear(treatment_dim, emb_dim),
            nn.LayerNorm(emb_dim),
            nn.GELU()
        )

        fusion_dim = vision_dim + emb_dim
        self.regression_head = nn.Sequential(
            nn.Linear(fusion_dim, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(p=dropout),
            nn.Linear(256, 64),
            nn.LayerNorm(64),
            nn.GELU(),
            nn.Dropout(p=dropout * 0.5),
            nn.Linear(64, 1),
            nn.ReLU()
        )

    def extract_visual_features(self, x):
        x = self.features(x)
        x = self.norm(x)
        x = self.permute(x)
        x = self.avgpool(x)
        return torch.flatten(x, 1)

    def forward(self, image, treatment_one_hot):
        vis = self.extract_visual_features(image)
        treat = self.treatment_net(treatment_one_hot)
        fused = torch.cat([vis, treat], dim=1)
        return self.regression_head(fused)

class MultimodalResNetRSL(nn.Module):
    def __init__(self, pretrained=True, treatment_dim=3, emb_dim=32, dropout=0.3):
        super().__init__()
        self.model_name = 'multimodal_resnet50'
        weights = ResNet50_Weights.IMAGENET1K_V2 if pretrained else None
        base_resnet = resnet50(weights=weights)

        self.conv1 = base_resnet.conv1
        self.bn1 = base_resnet.bn1
        self.relu = base_resnet.relu
        self.maxpool = base_resnet.maxpool
        self.layer1 = base_resnet.layer1
        self.layer2 = base_resnet.layer2
        self.layer3 = base_resnet.layer3
        self.layer4 = base_resnet.layer4
        self.avgpool = base_resnet.avgpool
        vision_dim = 2048

        self.treatment_net = nn.Sequential(
            nn.Linear(treatment_dim, emb_dim),
            nn.LayerNorm(emb_dim),
            nn.GELU()
        )

        fusion_dim = vision_dim + emb_dim
        self.regression_head = nn.Sequential(
            nn.Linear(fusion_dim, 512),
            nn.BatchNorm1d(512),
            nn.GELU(),
            nn.Dropout(p=dropout),
            nn.Linear(512, 128),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Dropout(p=dropout * 0.5),
            nn.Linear(128, 1),
            nn.ReLU()
        )

    def extract_visual_features(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x)
        return torch.flatten(x, 1)

    def forward(self, image, treatment_one_hot):
        vis = self.extract_visual_features(image)
        treat = self.treatment_net(treatment_one_hot)
        fused = torch.cat([vis, treat], dim=1)
        return self.regression_head(fused)

def freeze_backbone(model):
    if hasattr(model, 'features'):
        for p in model.features.parameters():
            p.requires_grad = False
    elif hasattr(model, 'layer1'):
        for l in [model.conv1, model.bn1, model.layer1, model.layer2, model.layer3, model.layer4]:
            for p in l.parameters():
                p.requires_grad = False

def unfreeze_backbone(model):
    for p in model.parameters():
        p.requires_grad = True

def get_optimizer_groups(model, backbone_lr=1e-5, head_lr=1e-3, weight_decay=1e-4):
    head_names = {'treatment_net', 'regression_head'}
    backbone_params, head_params = [], []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if any(h in name for h in head_names):
            head_params.append(p)
        else:
            backbone_params.append(p)
    return [
        {'params': backbone_params, 'lr': backbone_lr, 'weight_decay': weight_decay},
        {'params': head_params, 'lr': head_lr, 'weight_decay': weight_decay}
    ]

print('Multimodal RSL architectures initialized successfully.')


### 4. Training Engine with Huber Loss (Smooth L1) & Early Stopping
- **Loss:** Smooth L1 (Huber Loss, $\beta=1.0$) for robust error penalization.
- **Stage 1 (Warmup):** 5 epochs on fusion head with frozen backbone.
- **Stage 2 (Fine-Tuning):** 30 epochs end-to-end with Cosine Annealing learning rate schedule.


In [ ]:
def compute_metrics(y_true, y_pred):
    mae = float(np.mean(np.abs(y_true - y_pred)))
    rmse = float(np.sqrt(np.mean((y_true - y_pred) ** 2)))
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    ss_res = np.sum((y_true - y_pred) ** 2)
    r2 = float(1.0 - (ss_res / (ss_tot + 1e-8)))
    return {'mae': mae, 'rmse': rmse, 'r2': r2}

def train_rsl_model(model_name, total_epochs=35, warmup_epochs=5, patience=8):
    print('=' * 75)
    print(f'STARTING MULTIMODAL RSL TRAINING: {model_name.upper()}')
    print('=' * 75)

    model = (MultimodalResNetRSL() if model_name == 'resnet50' else MultimodalSwinRSL()).to(device)
    criterion = nn.SmoothL1Loss(beta=1.0)
    os.makedirs('checkpoints_rsl', exist_ok=True)
    best_ckpt = f'checkpoints_rsl/best_{model_name}_rsl.pth'

    history = {'train_loss': [], 'val_loss': [], 'train_mae': [], 'val_mae': [], 'val_rmse': [], 'val_r2': []}

    # Stage 1: Warmup
    if warmup_epochs > 0:
        print(f'>>> Stage 1: Fusion Head Warmup ({warmup_epochs} epochs, frozen backbone) <<<')
        freeze_backbone(model)
        optimizer_warmup = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3, weight_decay=1e-4)

        for ep in range(1, warmup_epochs + 1):
            model.train()
            t_loss, p_list, t_list, n_tot = 0.0, [], [], 0
            for imgs, treats, targets in train_loader:
                imgs, treats, targets = imgs.to(device), treats.to(device), targets.to(device)
                optimizer_warmup.zero_grad()
                outs = model(imgs, treats)
                loss = criterion(outs, targets)
                loss.backward()
                optimizer_warmup.step()

                bs = targets.size(0)
                t_loss += loss.item() * bs
                n_tot += bs
                p_list.append(outs.detach().cpu().numpy())
                t_list.append(targets.detach().cpu().numpy())

            tr_mae = compute_metrics(np.vstack(t_list).flatten(), np.vstack(p_list).flatten())['mae']

            model.eval()
            v_loss, vp_list, vt_list, vn_tot = 0.0, [], [], 0
            with torch.no_grad():
                for imgs, treats, targets in val_loader:
                    imgs, treats, targets = imgs.to(device), treats.to(device), targets.to(device)
                    outs = model(imgs, treats)
                    bs = targets.size(0)
                    v_loss += criterion(outs, targets).item() * bs
                    vn_tot += bs
                    vp_list.append(outs.cpu().numpy())
                    vt_list.append(targets.cpu().numpy())

            v_met = compute_metrics(np.vstack(vt_list).flatten(), np.vstack(vp_list).flatten())
            print(f'  [Warmup {ep:02d}/{warmup_epochs:02d}] Train Loss: {t_loss/n_tot:.4f}, MAE: {tr_mae:.2f}d || Val Loss: {v_loss/vn_tot:.4f}, MAE: {v_met["mae"]:.2f}d, R²: {v_met["r2"]:.3f}')

    # Stage 2: Fine-Tuning
    print(f'>>> Stage 2: Full Fine-Tuning ({total_epochs - warmup_epochs} epochs) <<<')
    unfreeze_backbone(model)
    optimizer = AdamW(get_optimizer_groups(model, backbone_lr=1e-5, head_lr=5e-4))
    scheduler = CosineAnnealingLR(optimizer, T_max=total_epochs - warmup_epochs, eta_min=1e-7)

    best_val_mae = float('inf')
    patience_cnt = 0

    for ep in range(1, total_epochs - warmup_epochs + 1):
        cur_ep = warmup_epochs + ep
        model.train()
        t_loss, p_list, t_list, n_tot = 0.0, [], [], 0
        for imgs, treats, targets in train_loader:
            imgs, treats, targets = imgs.to(device), treats.to(device), targets.to(device)
            optimizer.zero_grad()
            outs = model(imgs, treats)
            loss = criterion(outs, targets)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            bs = targets.size(0)
            t_loss += loss.item() * bs
            n_tot += bs
            p_list.append(outs.detach().cpu().numpy())
            t_list.append(targets.detach().cpu().numpy())
        scheduler.step()

        tr_mae = compute_metrics(np.vstack(t_list).flatten(), np.vstack(p_list).flatten())['mae']

        model.eval()
        v_loss, vp_list, vt_list, vn_tot = 0.0, [], [], 0
        with torch.no_grad():
            for imgs, treats, targets in val_loader:
                imgs, treats, targets = imgs.to(device), treats.to(device), targets.to(device)
                outs = model(imgs, treats)
                bs = targets.size(0)
                v_loss += criterion(outs, targets).item() * bs
                vn_tot += bs
                vp_list.append(outs.cpu().numpy())
                vt_list.append(targets.cpu().numpy())

        v_met = compute_metrics(np.vstack(vt_list).flatten(), np.vstack(vp_list).flatten())
        va_l = v_loss / vn_tot

        history['train_loss'].append(t_loss / n_tot)
        history['val_loss'].append(va_l)
        history['train_mae'].append(tr_mae)
        history['val_mae'].append(v_met['mae'])
        history['val_rmse'].append(v_met['rmse'])
        history['val_r2'].append(v_met['r2'])

        mark = ''
        if v_met['mae'] < best_val_mae:
            best_val_mae = v_met['mae']
            patience_cnt = 0
            torch.save({'state_dict': model.state_dict(), 'val_mae': best_val_mae, 'metrics': v_met}, best_ckpt)
            mark = ' [* BEST]'
        else:
            patience_cnt += 1

        print(f'  [Epoch {cur_ep:02d}/{total_epochs:02d}] Train Loss: {t_loss/n_tot:.4f}, MAE: {tr_mae:.2f}d || Val Loss: {va_l:.4f}, MAE: {v_met["mae"]:.2f}d, R²: {v_met["r2"]:.3f}{mark}')
        if patience_cnt >= patience:
            print(f'Early stopping triggered after {patience} epochs without improvement.')
            break

    print(f'Finished {model_name}. Best Val MAE: {best_val_mae:.2f} days | Checkpoint: {best_ckpt}')
    return history


### 5. Training Execution: Multimodal ResNet50 & Swin Transformer


In [ ]:
# Train Multimodal ResNet50
hist_resnet = train_rsl_model('resnet50', total_epochs=35, warmup_epochs=5, patience=8)

# Train Multimodal Swin-T
hist_swin = train_rsl_model('swin_t', total_epochs=35, warmup_epochs=5, patience=8)

# Plot Comparative Regression Learning Curves
os.makedirs('outputs_rsl', exist_ok=True)
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(hist_resnet['val_loss'], 'b-o', label='ResNet50 Val Loss')
plt.plot(hist_swin['val_loss'], 'r-s', label='Swin-T Val Loss')
plt.title('Validation Smooth L1 Loss', fontweight='bold')
plt.xlabel('Fine-Tuning Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, linestyle='--')

plt.subplot(1, 2, 2)
plt.plot(hist_resnet['val_mae'], 'b-o', label='ResNet50 Val MAE (Days)')
plt.plot(hist_swin['val_mae'], 'r-s', label='Swin-T Val MAE (Days)')
plt.title('Validation Mean Absolute Error (MAE in Days)', fontweight='bold')
plt.xlabel('Fine-Tuning Epoch')
plt.ylabel('MAE (Days)')
plt.legend()
plt.grid(True, linestyle='--')

plt.tight_layout()
plt.savefig('outputs_rsl/rsl_training_curves_comparison.png', dpi=200)
plt.show()


### 6. Held-Out Test Set Evaluation & Actual vs. Predicted Plots
Evaluates unseen physical mushroom samples (IDs: 1, 4, 21) across all storage days.


In [ ]:
def evaluate_test_rsl(model_name):
    model = (MultimodalResNetRSL(pretrained=False) if model_name == 'resnet50' else MultimodalSwinRSL(pretrained=False)).to(device)
    ckpt = torch.load(f'checkpoints_rsl/best_{model_name}_rsl.pth', map_location=device)
    model.load_state_dict(ckpt['state_dict'])
    model.eval()

    y_true, y_pred = [], []
    with torch.no_grad():
        for imgs, treats, targets in test_loader:
            imgs, treats = imgs.to(device), treats.to(device)
            outs = model(imgs, treats)
            y_true.extend(targets.squeeze().cpu().numpy())
            y_pred.extend(outs.squeeze().cpu().numpy())

    y_true, y_pred = np.array(y_true), np.array(y_pred)
    metrics = compute_metrics(y_true, y_pred)

    print('=' * 60)
    print(f'{model_name.upper()} FINAL TEST SET PERFORMANCE:')
    print(f'  - Mean Absolute Error (MAE):  {metrics["mae"]:.2f} days')
    print(f'  - Root Mean Squared Err (RMSE): {metrics["rmse"]:.2f} days')
    print(f'  - Coefficient of Det (R²):     {metrics["r2"]:.4f}')
    print('=' * 60)

    # Plot Scatter Actual vs. Predicted
    plt.figure(figsize=(7, 6))
    plt.scatter(y_true, y_pred, color='#2ca02c' if model_name == 'swin_t' else '#1f77b4',
                alpha=0.7, edgecolors='k', s=65, label='Test Samples (IDs: 1, 4, 21)')
    max_val = max(float(y_true.max()), float(y_pred.max()), 20.0)
    plt.plot([0, max_val], [0, max_val], 'r--', linewidth=2, label='Ideal Fit (y = x)')
    plt.xlabel('Actual Remaining Shelf Life (Days)', fontweight='bold', fontsize=11)
    plt.ylabel('Predicted Remaining Shelf Life (Days)', fontweight='bold', fontsize=11)
    plt.title(f'{model_name.upper()} RSL Prediction on Test Set\nMAE: {metrics["mae"]:.2f}d | RMSE: {metrics["rmse"]:.2f}d | R²: {metrics["r2"]:.3f}', fontweight='bold')
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.legend()
    plt.tight_layout()
    plt.savefig(f'outputs_rsl/{model_name}_test_actual_vs_pred.png', dpi=200)
    plt.show()

evaluate_test_rsl('resnet50')
evaluate_test_rsl('swin_t')
